# **Notebook 05 — Modelling: XGBoost**

## Objectives

* Compare baseline algorithms (Random Forest, Gradient Boosting, XGBoost)
* Perform hyperparameter optimisation with 8+ parameters and rationale
* Evaluate model on train and test sets (confusion matrix, classification report)
* Generate SHAP explainability artifacts
* Validate Hypothesis H4

## Inputs

* `outputs/v1/X_train_resampled.csv`, `outputs/v1/y_train_resampled.csv`
* `outputs/v1/X_test_engineered.csv`, `outputs/v1/y_test.csv`

## Outputs

* `outputs/v1/xgb_baseline.pkl` — baseline model
* `outputs/v2/fraud_model_optimized.pkl` — optimised model
* `outputs/v2/shap_explainer.pkl` — SHAP explainer
* `outputs/v2/optimal_threshold.json` — best threshold
* `outputs/v2/evaluation_report.json` — metrics summary

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\name\Desktop\All IN\Code Inst. Projects\credit-card-fraud-detection


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json

X_train = pd.read_csv("outputs/v1/X_train_resampled.csv")
y_train = pd.read_csv("outputs/v1/y_train_resampled.csv").squeeze()
X_test = pd.read_csv("outputs/v1/X_test_engineered.csv")
y_test = pd.read_csv("outputs/v1/y_test.csv").squeeze()

print(f"Train set: {X_train.shape} (SMOTE balanced)")
print(f"Test set:  {X_test.shape} (original distribution)")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test fraud cases:    {y_test.sum()}")

---
## 1. Algorithm Comparison

Comparing three ensemble classifiers to select the best base algorithm before hyperparameter tuning. Using 5-fold stratified cross-validation for fair comparison.

In [ ]:
! pip install xgboost

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, random_state=42,
        eval_metric='logloss', use_label_encoder=False
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Algorithm Comparison (5-fold Stratified CV)")
print("=" * 60)

comparison_results = {}
for name, model in models.items():
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    pr_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='precision')
    rc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='recall')

    comparison_results[name] = {
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'precision': pr_scores.mean(),
        'recall': rc_scores.mean()
    }

    print(f"\n{name}:")
    print(f"  F1:        {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})")
    print(f"  Precision: {pr_scores.mean():.4f}")
    print(f"  Recall:    {rc_scores.mean():.4f}")

---
## 1. Algorithm Comparison

Comparing three ensemble classifiers to select the best base algorithm before hyperparameter tuning. Using 5-fold stratified cross-validation for fair comparison.

### Algorithm Selection Rationale

XGBoost selected as the primary algorithm because:
1. **Highest F1 score** in cross-validation among the three candidates
2. **Best precision-recall balance** for imbalanced fraud detection
3. **Built-in regularisation** (gamma, lambda) prevents overfitting on SMOTE data
4. **scale_pos_weight** parameter for additional imbalance handling
5. **Feature importance** readily available for SHAP integration
6. **Fast inference** — important for real-time transaction scoring

---
## 2. Save Baseline Model

Saving the default XGBoost model before tuning, so we can compare baseline vs optimised performance.

In [ ]:
baseline = XGBClassifier(
    n_estimators=100, random_state=42,
    eval_metric='logloss', use_label_encoder=False
)
baseline.fit(X_train, y_train)

os.makedirs("outputs/v1", exist_ok=True)
joblib.dump(baseline, "outputs/v1/xgb_baseline.pkl")
print("Baseline model saved to outputs/v1/xgb_baseline.pkl")

---
## 3. Hyperparameter Tuning (Distinction)

Using RandomizedSearchCV with **8 hyperparameters** and **3+ values each**, with documented rationale for every choice.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.3],
    'scale_pos_weight': [1, 10, 50],
}

print("Hyperparameter Search Space")
print("=" * 50)
for param, values in param_distributions.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations: {np.prod([len(v) for v in param_distributions.values()]):,}")

### Hyperparameter Tuning Rationale

| Parameter | Values | Rationale |
|-----------|--------|-----------|
| n_estimators | 100, 300, 500 | More boosting rounds improve ensemble strength. Upper limit of 500 balances performance vs computation time. Works together with learning_rate for optimal convergence. |
| max_depth | 3, 5, 7 | Shallow trees (3-7) prevent overfitting. Critical for SMOTE data where deep trees may memorise synthetic samples rather than learning generalisable fraud patterns. |
| learning_rate | 0.01, 0.05, 0.1 | Step size shrinkage per round. Lower rates produce more robust models but need more estimators. Range covers conservative (0.01) to moderately aggressive (0.1). |
| min_child_weight | 1, 3, 5 | Minimum sum of instance weight in a leaf. Higher values create more conservative splits, preventing overly specific rules for rare fraud patterns. |
| subsample | 0.7, 0.8, 0.9 | Fraction of training samples per tree. Row subsampling adds randomness to reduce variance. Not below 0.7 to keep enough signal from the fraud class. |
| colsample_bytree | 0.7, 0.8, 0.9 | Fraction of features per tree. Prevents over-reliance on dominant features like V14. Forces the model to learn from diverse feature combinations. |
| gamma | 0, 0.1, 0.3 | Minimum loss reduction for a split. Higher values prune the tree more aggressively. Range from no constraint (0) to moderate pruning (0.3). |
| scale_pos_weight | 1, 10, 50 | Additional class weighting on top of SMOTE. Value of 50 approximates the original imbalance ratio for maximum fraud recall. |

In [1]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

search = RandomizedSearchCV(
    xgb, param_distributions,
    n_iter=50,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(f"\nBest F1 (CV): {search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

NameError: name 'XGBClassifier' is not defined